In [ ]:
# part_c_lvq_playtennis.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

try:
    from sklearn_lvq import GlvqModel
except Exception as e:
    raise ImportError("sklearn_lvq not found. Install with: pip install sklearn-lvq") from e

# Play Tennis dataset (manually entered)
data = [
    ["Sunny","Hot","High","Weak","No"],
    ["Sunny","Hot","High","Strong","No"],
    ["Overcast","Hot","High","Weak","Yes"],
    ["Rain","Mild","High","Weak","Yes"],
    ["Rain","Cool","Normal","Weak","Yes"],
    ["Rain","Cool","Normal","Strong","No"]
]
df = pd.DataFrame(data, columns=["Outlook","Temperature","Humidity","Wind","PlayTennis"])

# Features + label
X = df[["Outlook","Temperature","Humidity","Wind"]]
y = df["PlayTennis"]

# Label encode categorical features
encoders = {}
X_enc = pd.DataFrame()
for col in X.columns:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X[col])
    encoders[col] = le

# Encode label
le_label = LabelEncoder()
y_enc = le_label.fit_transform(y)  # 0/1 for No/Yes

# Normalize to [0,1]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_enc)

# split (80/20). Because dataset is tiny, stratify may fail — here we do a simple split.
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc if len(np.unique(y_enc))>1 else None)

# train GLVQ
model = GlvqModel()
model.fit(X_train, y_train)

# eval
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"PlayTennis test accuracy (GLVQ): {acc*100:.2f}%")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le_label.classes_)
disp.plot(values_format='d')
plt.title("Confusion matrix — GLVQ on PlayTennis")
plt.show()
